In [1]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def nettoyer_et_separer_donnees(fichier_entree, nom_feuille='Débit non null'):
    # 1. Chargement des données brutes
    df = pd.read_excel(fichier_entree, sheet_name=nom_feuille)
    
    # Identification dynamique des colonnes
    colonnes_totaliseurs = [col for col in df.columns if 'Totaliseur' in col]
    colonnes_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Initialisation des colonnes de suivi des anomalies
    df['Est_Anomalie'] = False
    df['Motif_Anomalie'] = ""
    
    # Règle 1 : Présence d'au moins une case vide (NaN)
    lignes_vides = df.isnull().any(axis=1)
    df.loc[lignes_vides, 'Est_Anomalie'] = True
    df.loc[lignes_vides, 'Motif_Anomalie'] += "Case vide; "
    
    # Règle 2 : Présence d'au moins une valeur négative
    lignes_negatives = (df[colonnes_numeriques] < 0).any(axis=1)
    df.loc[lignes_negatives, 'Est_Anomalie'] = True
    df.loc[lignes_negatives, 'Motif_Anomalie'] += "Valeur négative; "
    
    # Règles 3 & 4 : Analyse séquentielle ligne par ligne
    for i in range(1, len(df)):
        time_curr = df.loc[i, 'Time']
        time_prev = df.loc[i-1, 'Time']
        
        # AJUSTEMENT : Vérification du pas de temps de 40 minutes UNIQUEMENT pour le même jour
        if time_curr.date() == time_prev.date():
            diff_temps = (time_curr - time_prev).total_seconds() / 60.0
            if diff_temps != 40.0:
                df.loc[i, 'Est_Anomalie'] = True
                df.loc[i, 'Motif_Anomalie'] += f"Pas de temps incorrect même jour ({int(diff_temps) if diff_temps.is_integer() else diff_temps} min); "
            
        # Règle 4 : Vérification de la décroissance d'un totaliseur
        for col in colonnes_totaliseurs:
            val_actuelle = df.loc[i, col]
            val_precedente = df.loc[i-1, col]
            if pd.notna(val_actuelle) and pd.notna(val_precedente):
                if val_actuelle < val_precedente:
                    df.loc[i, 'Est_Anomalie'] = True
                    df.loc[i, 'Motif_Anomalie'] += f"Diminution du totaliseur ({col}); "
                    
    # Nettoyage de la chaîne de caractères des motifs
    df['Motif_Anomalie'] = df['Motif_Anomalie'].str.rstrip('; ')
    
    # Séparation finale des DataFrames
    df_nettoyees = df[df['Est_Anomalie'] == False].drop(columns=['Est_Anomalie', 'Motif_Anomalie'])
    df_anomalies = df[df['Est_Anomalie'] == True].copy()
    
    return df_nettoyees, df_anomalies

def sauvegarder_excel_style(df, nom_fichier, nom_feuille, couleur_entete="2C3E50", accent_row_highlight=None):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = nom_feuille
    ws.views.sheetView[0].showGridLines = True
    
    # Styles graphiques
    police_entete = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
    police_corps = Font(name='Segoe UI', size=10, bold=False, color='333333')
    police_alerte = Font(name='Segoe UI', size=10, bold=True, color='9C0006')
    
    remplissage_entete = PatternFill(start_color=couleur_entete, end_color=couleur_entete, fill_type='solid')
    remplissage_zebra = PatternFill(start_color='F8F9FA', end_color='F8F9FA', fill_type='solid')
    remplissage_alerte = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    
    bordure_fine = Border(
        left=Side(style='thin', color='E0E0E0'), right=Side(style='thin', color='E0E0E0'),
        top=Side(style='thin', color='E0E0E0'), bottom=Side(style='thin', color='E0E0E0')
    )
    
    # En-têtes
    en_tetes = df.columns.tolist()
    ws.append(en_tetes)
    for col_num, header in enumerate(en_tetes, 1):
        cellule = ws.cell(row=1, column=col_num)
        cellule.font = police_entete
        cellule.fill = remplissage_entete
        cellule.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cellule.border = bordure_fine
    
    # Remplissage des lignes
    for ligne_idx, ligne_data in enumerate(df.values, 2):
        valeurs_ligne = [None if pd.isna(v) else v for v in ligne_data]
        ws.append(valeurs_ligne)
        
        for col_num in range(1, len(en_tetes) + 1):
            cellule = ws.cell(row=ligne_idx, column=col_num)
            cellule.font = police_corps
            cellule.border = bordure_fine
            
            val = cellule.value
            if isinstance(val, (int, float)):
                cellule.alignment = Alignment(horizontal='right', vertical='center')
                cellule.number_format = '#,##0.00'
            elif isinstance(val, pd.Timestamp) or hasattr(val, 'strftime'):
                cellule.alignment = Alignment(horizontal='center', vertical='center')
                cellule.number_format = 'yyyy-mm-dd hh:mm:ss'
            else:
                cellule.alignment = Alignment(horizontal='left', vertical='center')
                
            if ligne_idx % 2 == 0 and en_tetes[col_num-1] != 'Motif_Anomalie':
                cellule.fill = remplissage_zebra
                
            if en_tetes[col_num-1] == 'Motif_Anomalie' and val:
                cellule.fill = remplissage_alerte
                cellule.font = police_alerte
                
    # Ajustement automatique de la largeur des colonnes
    ws.row_dimensions[1].height = 28
    for col in ws.columns:
        max_len = 0
        lettre_col = get_column_letter(col[0].column)
        for cellule in col:
            if cellule.value:
                val_str = cellule.value.strftime('%Y-%m-%d %H:%M:%S') if isinstance(cellule.value, pd.Timestamp) else str(cellule.value)
                if len(val_str) > max_len:
                    max_len = len(val_str)
        ws.column_dimensions[lettre_col].width = max(max_len + 4, 12)
        
    wb.save(nom_fichier)

# --- SCRIPT D'ÉXÉCUTION ---
if __name__ == "__main__":
    fichier_brut = 'Data petcock for ai (1).xlsx'
    
    # Nettoyage et tri
    df_nettoyees, df_anomalies = nettoyer_et_separer_donnees(fichier_brut)
    
    # Sauvegarde des versions V2 stylisées
    sauvegarder_excel_style(df_nettoyees, 'Donnees_Nettoyees_v2.xlsx', 'Data Nettoyée', couleur_entete='1E4620')
    sauvegarder_excel_style(df_anomalies, 'Donnees_Anomalies_v2.xlsx', 'Anomalies', couleur_entete='7D1C1C', accent_row_highlight=True)
    
    print("Fichiers V2 générés avec succès.")

Fichiers V2 générés avec succès.


In [2]:
import pandas as pd

# 1. Chargement des fichiers Excel générés précédemment
fichier_propres = 'Donnees_Nettoyees_v2.xlsx'
fichier_anomalies = 'Donnees_Anomalies_v2.xlsx'

df_nettoyees = pd.read_excel(fichier_propres)
df_anomalies = pd.read_excel(fichier_anomalies)

# 2. Affichage du nombre de lignes de chaque tableau
print("=" * 60)
print(f"📊 NOMBRE DE LIGNES CRÉÉES :")
print(f"   • Tableau des données nettoyées : {len(df_nettoyees)} lignes")
print(f"   • Tableau des anomalies détectées : {len(df_anomalies)} lignes")
print("=" * 60)
print("\n")

# 3. Affichage du Tableau des Anomalies (Complet - 17 lignes)
print("🚨 --- TABLEAU DES ANOMALIES (AFFICHAGE COMPLET) ---")
# Pour l'affichage console, on sélectionne les colonnes clés pour que ce soit lisible
colonnes_visibles_anom = ['Time', 'Motif_Anomalie'] 
print(df_anomalies[colonnes_visibles_anom].to_string(index=False))
print("\n" + "-" * 60 + "\n")

# 4. Affichage du Tableau des Données Nettoyées (Extrait - 10 premières lignes)
print("✨ --- TABLEAU DES DONNÉES NETTOYÉES (EXTRAIT - 10 PREMIÈRES LIGNES) ---")
# Sélection de quelques colonnes pour l'exemple d'affichage
colonnes_visibles_propres = [
    'Time', 
    'Totaliseur injection coke en kg', 
    'Consigne injection petcock en t/h'
]
print(df_nettoyees[colonnes_visibles_propres].head(10).to_string(index=False))
print(f"\n... ({len(df_nettoyees) - 10} lignes restantes masquées dans l'affichage console) ...")

📊 NOMBRE DE LIGNES CRÉÉES :
   • Tableau des données nettoyées : 69 lignes
   • Tableau des anomalies détectées : 17 lignes


🚨 --- TABLEAU DES ANOMALIES (AFFICHAGE COMPLET) ---
               Time                                                                                                               Motif_Anomalie
2026-04-15 14:00:00                                                                                   Pas de temps incorrect même jour (240 min)
2026-04-15 18:00:00                                                                                    Pas de temps incorrect même jour (80 min)
2026-04-16 21:20:00                                                                                   Pas de temps incorrect même jour (160 min)
2026-04-17 12:40:00 Case vide; Diminution du totaliseur (Totaliseur fuel P2 en kg); Diminution du totaliseur (Totaliseur de production RS3 en t)
2026-04-17 13:20:00                                                                              